# T09 — Train PhoBERT phần B trên dữ liệu phim

Notebook này chạy trên Google Colab GPU. Mọi lựa chọn cấu hình/hyperparameter dùng duy nhất validation 15% tách phân tầng từ `movie_train.csv` với seed 42. Hai test set chỉ được đọc sau khi selection manifest đã khóa.

- Cấu hình a: chỉ movie train.
- Cấu hình b: movie train + mẫu phone từ phone train/dev theo tỷ lệ số dòng 1:1.
- Checkpoint khởi tạo bắt buộc: research `phobert_A_best`, không dùng checkpoint backend.
- Nhãn: `0 = binh_thuong`, `1 = tieu_cuc`; `max_length=256`; preprocessing là `normalize()` production hiện hành.
- Không chạy lại hoặc đổi selection sau khi đã chạy phần **Final test — chạy đúng một lần**.

## 1. Runtime và dependency

Trong Colab chọn **Runtime → Change runtime type → T4 GPU**. Cell chỉ pin phiên bản Transformers đã chốt của dự án; không thêm dependency project mới. Sau khi cài khác phiên bản đang nạp, Colab có thể yêu cầu restart runtime rồi chạy lại từ đầu.

In [ ]:
import importlib.metadata as metadata, subprocess, sys
required = {'transformers': '5.18.0', 'scikit-learn': '1.7.2', 'joblib': '1.5.2'}
for package, version in required.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'{package}=={version}'])
print({name: metadata.version(name) for name in required})

## 2. Mount Drive và tìm repo/dữ liệu có kiểm soát

Notebook không phụ thuộc current working directory. Có thể đặt biến môi trường `FIREFLY_PROJECT_ROOT`; nếu không, notebook tự tìm repo trong thư mục hiện tại, các thư mục cha, rồi `/content/firefly-cinema`. Dữ liệu/model lớn có thể nằm trong repo local ngoài Git hoặc đúng các đường dẫn Drive bên dưới.

In [ ]:
from pathlib import Path
import os, shutil, sys

if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')

root_candidates = []
if os.environ.get('FIREFLY_PROJECT_ROOT'):
    root_candidates.append(Path(os.environ['FIREFLY_PROJECT_ROOT']))
cwd = Path.cwd().resolve()
root_candidates.extend([cwd, *cwd.parents, Path('/content/firefly-cinema')])
PROJECT_ROOT = next((p.resolve() for p in root_candidates if (p / 'ml/src/model_b_utils.py').is_file()), None)
code_archive = Path('/content/drive/MyDrive/thesis/code/firefly-cinema-T09-code.zip')
if PROJECT_ROOT is None and code_archive.is_file():
    shutil.unpack_archive(code_archive, '/content/firefly-cinema')
    PROJECT_ROOT = Path('/content/firefly-cinema').resolve()
assert PROJECT_ROOT is not None, 'Không tìm thấy repo; đặt FIREFLY_PROJECT_ROOT hoặc clone/upload vào /content/firefly-cinema'
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'ml/src'))
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
import json, math, shutil, time
from copy import deepcopy

import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

# Chỉ import normalize runtime; chặn registry tải model backend ngoài phạm vi T09.
os.environ['MODEL_DIR'] = '__t09_normalize_only_missing__'
os.environ['SKLEARN_MODELS'] = ''
from backend.app.classifier import normalize as runtime_normalize
from eval_utils import normalize
from model_b_utils import (
    LABELS, METRIC_COLUMNS, SEED, assert_comparison_schema, build_training_configs,
    comparison_rows, constant_positive_metrics, fit_selected_traditional_models,
    load_movie_train, load_phone_pool, metrics, seed_everything,
    select_traditional_models, selection_manifest, sha256, stratified_movie_split, write_json,
)

seed_everything(SEED)
assert torch.cuda.is_available(), 'T09 PhoBERT phải chạy bằng Colab GPU'
DEVICE = torch.device('cuda')
DRIVE_ROOT = Path('/content/drive/MyDrive')

def first_dir_with_files(required, *paths):
    found = next((Path(p) for p in paths if all((Path(p) / name).is_file() for name in required)), None)
    assert found is not None, 'Thiếu artifact ' + str(required) + '; đã tìm: ' + ', '.join(map(str, paths))
    return found

MOVIE_DIR = first_dir_with_files(('movie_train.csv', 'movie_test.csv'), PROJECT_ROOT / 'data/phim', DRIVE_ROOT / 'thesis/data/movie', DRIVE_ROOT / 'thesis/data/phim')
PHONE_DIR = first_dir_with_files(('train.csv', 'dev.csv', 'test.csv'), PROJECT_ROOT / 'data/uit-visfd/processed', DRIVE_ROOT / 'thesis/data/processed')
RESEARCH_CKPT = first_dir_with_files(('model.safetensors', 'config.json'), PROJECT_ROOT / 'ml/artifacts/phobert_A_best', DRIVE_ROOT / 'thesis_results/phobert_A_best')
BASELINE_A_DIR = first_dir_with_files(('svm_A.joblib', 'lr_A.joblib', 'nb_A.joblib'), PROJECT_ROOT / 'ml/artifacts/baselines', DRIVE_ROOT / 'thesis_results/baselines')
MOVIE_TRAIN_PATH = MOVIE_DIR / 'movie_train.csv'
PHONE_TRAIN_PATH = PHONE_DIR / 'train.csv'
PHONE_DEV_PATH = PHONE_DIR / 'dev.csv'
for path in [MOVIE_TRAIN_PATH, PHONE_TRAIN_PATH, PHONE_DEV_PATH, RESEARCH_CKPT / 'model.safetensors']:
    assert path.exists(), f'Missing: {path}'
assert 'backend/models' not in RESEARCH_CKPT.as_posix(), 'Không được fine-tune từ checkpoint backend'
RESULT_DIR = PROJECT_ROOT / 'ml/results'
LOCAL_B_DIR = PROJECT_ROOT / 'ml/artifacts/phobert_B_best'
LOCAL_TRAD_DIR = PROJECT_ROOT / 'ml/artifacts/baselines_B'
DRIVE_B_DIR = DRIVE_ROOT / 'thesis_results/phobert_B_best'
DRIVE_RESULT_DIR = DRIVE_ROOT / 'thesis_results/T09'
for path in [RESULT_DIR, LOCAL_B_DIR, LOCAL_TRAD_DIR, DRIVE_B_DIR, DRIVE_RESULT_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print('GPU =', torch.cuda.get_device_name(0))
print('RESEARCH_CKPT =', RESEARCH_CKPT)
print('research_sha256 =', sha256(RESEARCH_CKPT / 'model.safetensors'))

## 3. Gate preprocessing, dữ liệu và split validation

Cell này so sánh `normalize()` dùng cho train/eval với runtime trên 20 câu (10 phone + 10 movie), sau đó mới tạo validation 15% phân tầng. Phone test chưa được đọc.

In [ ]:
movie_all = load_movie_train(MOVIE_TRAIN_PATH)
phone_pool = load_phone_pool(PHONE_TRAIN_PATH, PHONE_DEV_PATH)
parity_texts = movie_all['text'].head(10).astype(str).tolist() + phone_pool['text'].head(10).astype(str).tolist()
parity = pd.DataFrame({'text': parity_texts})
parity['train_eval'] = parity['text'].map(normalize)
parity['runtime'] = parity['text'].map(runtime_normalize)
parity['equal'] = parity['train_eval'] == parity['runtime']
display(parity)
assert len(parity) == 20 and parity['equal'].all()
assert LABELS == {0: 'binh_thuong', 1: 'tieu_cuc'}
movie_train_part, movie_validation = stratified_movie_split(movie_all)
configs = build_training_configs(movie_train_part, phone_pool, normalize)
validation_norm = movie_validation.copy()
validation_norm['text'] = validation_norm['text'].map(normalize)
summary = {
    'movie_all': movie_all['label'].value_counts().sort_index().to_dict(),
    'movie_train_part': movie_train_part['label'].value_counts().sort_index().to_dict(),
    'movie_validation': movie_validation['label'].value_counts().sort_index().to_dict(),
    'config_a_rows': len(configs['a']), 'config_b_rows': len(configs['b']),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert len(movie_validation) == 53
assert len(configs['a']) == len(movie_train_part) and len(configs['b']) == 2 * len(movie_train_part)
print('NORMALIZE_PARITY_OK=20/20; TEST_NOT_LOADED=True')

## 4. Chọn SVM/LR/NB bằng movie validation

Mỗi mô hình thử cả a/b và cùng grid T01. Quy tắc chọn: macro-F1 validation cao nhất, rồi F1 tiêu cực; hòa tuyệt đối ưu tiên a. Sau khi chọn, mô hình được fit lại trên toàn bộ `movie_train`; cấu hình b lấy lại mẫu phone 1:1 với seed 42.

In [ ]:
traditional_selections, traditional_trials = select_traditional_models(configs, movie_validation, normalize)
traditional_summary = pd.DataFrame([
    {'model': name, 'train_config': value['train_config'], **value['params'], **value['validation_metrics']}
    for name, value in traditional_selections.items()
]).sort_values('model')
display(traditional_summary)
final_configs = build_training_configs(movie_all, phone_pool, normalize)
traditional_B_models = fit_selected_traditional_models(traditional_selections, final_configs, LOCAL_TRAD_DIR)
print('TRADITIONAL_SELECTION_LOCKED; TEST_NOT_LOADED=True')

## 5. Fine-tune PhoBERT từ checkpoint research

Thử a/b × learning rate `1e-5`, `2e-5`, tối đa 5 epoch, early stopping patience 2. Mỗi trial luôn nạp mới từ `RESEARCH_CKPT`. Metric chọn là macro-F1 trên movie validation, sau đó F1 tiêu cực.

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 5
PATIENCE = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
LEARNING_RATES = (1e-5, 2e-5)
TRIAL_ROOT = Path('/content/t09_trials')
TRIAL_ROOT.mkdir(parents=True, exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(RESEARCH_CKPT, local_files_only=True)

class TextDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame['text'].astype(str).tolist()
        self.labels = frame['label'].astype(int).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, index): return self.texts[index], self.labels[index]

def collate(batch):
    texts, labels = zip(*batch)
    encoded = tokenizer(list(texts), padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
    encoded['labels'] = torch.tensor(labels, dtype=torch.long)
    return encoded

def predict_model(model, frame, batch_size=16):
    loader = DataLoader(TextDataset(frame), batch_size=batch_size, shuffle=False, collate_fn=collate)
    predictions = []
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            labels = batch.pop('labels')
            logits = model(**{key: value.to(DEVICE) for key, value in batch.items()}).logits
            predictions.extend(torch.argmax(logits, dim=-1).cpu().tolist())
    return np.asarray(predictions, dtype=int)

def train_trial(config_name, learning_rate):
    seed_everything(SEED)
    training = configs[config_name]
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(TextDataset(training), batch_size=BATCH_SIZE, shuffle=True, generator=generator, collate_fn=collate)
    model = AutoModelForSequenceClassification.from_pretrained(RESEARCH_CKPT, local_files_only=True, num_labels=2)
    model.config.id2label = LABELS.copy()
    model.config.label2id = {value: key for key, value in LABELS.items()}
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * len(loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=True)
    trial_dir = TRIAL_ROOT / f'{config_name}_lr_{learning_rate:.0e}'
    best_score, best_epoch, stale = (-1.0, -1.0), 0, 0
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train(); total_loss = 0.0
        for batch in loader:
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                loss = model(**batch).loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            total_loss += float(loss.detach().cpu())
        validation_predictions = predict_model(model, validation_norm)
        score = metrics(validation_norm['label'], validation_predictions)
        history.append({'epoch': epoch, 'train_loss': total_loss / len(loader), **score})
        selection_score = (score['macro_f1'], score['f1_neg'])
        print(config_name, learning_rate, history[-1])
        if selection_score > best_score:
            best_score, best_epoch, stale = selection_score, epoch, 0
            model.save_pretrained(trial_dir)
            tokenizer.save_pretrained(trial_dir)
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()
    return {
        'train_config': config_name, 'learning_rate': learning_rate, 'best_epoch': best_epoch,
        'validation_metrics': {'macro_f1': best_score[0], 'f1_neg': best_score[1]},
        'trial_dir': str(trial_dir), 'history': history,
    }

In [ ]:
phobert_trials = []
for train_config in ('a', 'b'):
    for learning_rate in LEARNING_RATES:
        phobert_trials.append(train_trial(train_config, learning_rate))
phobert_validation = pd.DataFrame([{
    'train_config': row['train_config'], 'learning_rate': row['learning_rate'],
    'best_epoch': row['best_epoch'], **row['validation_metrics']
} for row in phobert_trials])
display(phobert_validation)
phobert_selection = max(
    phobert_trials,
    key=lambda row: (row['validation_metrics']['macro_f1'], row['validation_metrics']['f1_neg'], row['train_config'] == 'a', -row['learning_rate']),
)
print('SELECTED_PHOBERT=', json.dumps({k: v for k, v in phobert_selection.items() if k != 'history'}, ensure_ascii=False, indent=2))

## 6. Khóa selection trước khi đọc test

Cell này lưu model thắng validation và `selection_manifest.json`. Sau marker `SELECTION_LOCKED`, không sửa cấu hình, learning rate, epoch, preprocessing hay threshold dựa trên test.

In [ ]:
selected_trial_dir = Path(phobert_selection['trial_dir'])
selected_model = AutoModelForSequenceClassification.from_pretrained(selected_trial_dir, local_files_only=True)
selected_model.config.id2label = LABELS.copy()
selected_model.config.label2id = {value: key for key, value in LABELS.items()}
for destination in (LOCAL_B_DIR, DRIVE_B_DIR):
    selected_model.save_pretrained(destination)
    tokenizer.save_pretrained(destination)
manifest_phobert = {key: value for key, value in phobert_selection.items() if key not in {'history', 'trial_dir'}}
manifest = selection_manifest(
    MOVIE_TRAIN_PATH, PHONE_TRAIN_PATH, PHONE_DEV_PATH, movie_train_part, movie_validation,
    traditional_selections, manifest_phobert,
)
write_json(LOCAL_B_DIR / 'selection_manifest.json', manifest)
write_json(DRIVE_B_DIR / 'selection_manifest.json', manifest)
assert manifest['test_loaded'] is False
print('SELECTION_LOCKED=True; TEST_NOT_LOADED=True')

## 7. Final test — chạy đúng một lần

Chỉ cell này mới đọc `movie_test.csv` và phone `test.csv`. Bảng gồm model A canonical của T08, model B đã khóa, ba baseline A, ba baseline B đã chọn bằng validation, và baseline dự đoán hằng nhãn 1 để công bố ảnh hưởng lệch nhãn. Không dùng kết quả bên dưới để chọn lại.

In [ ]:
# TEST BOUNDARY: không chuyển việc đọc hai file này lên cell trước.
MOVIE_TEST_PATH = MOVIE_DIR / 'movie_test.csv'
PHONE_TEST_PATH = PHONE_DIR / 'test.csv'
assert MOVIE_TEST_PATH.exists() and PHONE_TEST_PATH.exists()
locked_manifest = json.loads((LOCAL_B_DIR / 'selection_manifest.json').read_text(encoding='utf-8'))
assert locked_manifest['test_loaded'] is False
movie_test_raw = pd.read_csv(MOVIE_TEST_PATH, encoding='utf-8-sig')
phone_test_raw = pd.read_csv(PHONE_TEST_PATH, encoding='utf-8-sig')
assert movie_test_raw.shape[0] == 149 and movie_test_raw['label'].value_counts().sort_index().to_dict() == {0: 48, 1: 101}
assert phone_test_raw.shape[0] == 2224 and phone_test_raw['has_negative'].value_counts().sort_index().to_dict() == {0: 981, 1: 1243}
test_frames = {
    'phone': pd.DataFrame({'text': phone_test_raw['text_raw'].map(normalize), 'label': phone_test_raw['has_negative'].astype(int)}),
    'movie': pd.DataFrame({'text': movie_test_raw['text'].map(normalize), 'label': movie_test_raw['label'].astype(int)}),
}
labels_by_test = {name: frame['label'].tolist() for name, frame in test_frames.items()}
rows = []

research_A = AutoModelForSequenceClassification.from_pretrained(RESEARCH_CKPT, local_files_only=True).to(DEVICE)
selected_model = AutoModelForSequenceClassification.from_pretrained(LOCAL_B_DIR, local_files_only=True).to(DEVICE)
for model_name, train_config, model in [
    ('PhoBERT A', 'A', research_A), ('PhoBERT B', phobert_selection['train_config'], selected_model)
]:
    predictions = {name: predict_model(model, frame) for name, frame in test_frames.items()}
    rows.extend(comparison_rows(model_name, train_config, labels_by_test, predictions))

baseline_A_files = {'SVM': 'svm_A.joblib', 'Logistic Regression': 'lr_A.joblib', 'Naive Bayes': 'nb_A.joblib'}
for name, filename in baseline_A_files.items():
    model = joblib.load(BASELINE_A_DIR / filename)
    predictions = {test: model.predict(frame['text']) for test, frame in test_frames.items()}
    rows.extend(comparison_rows(f'{name} A', 'A', labels_by_test, predictions))
for name, model in traditional_B_models.items():
    predictions = {test: model.predict(frame['text']) for test, frame in test_frames.items()}
    rows.extend(comparison_rows(f'{name} B', traditional_selections[name]['train_config'], labels_by_test, predictions))
for test_set in ('phone', 'movie'):
    score = constant_positive_metrics(labels_by_test[test_set])
    rows.append({'model': 'Constant positive', 'train_config': 'none', 'test_set': test_set, **{key: round(score[key], 6) for key in METRIC_COLUMNS}})
comparison = pd.DataFrame(rows, columns=['model', 'train_config', 'test_set', *METRIC_COLUMNS])
assert_comparison_schema(comparison)
t08_path = PROJECT_ROOT / 'ml/results/domain_shift.csv'
if t08_path.exists():
    t08 = pd.read_csv(t08_path).set_index(['model', 'test_set'])
    a_rows = comparison[comparison['model'].str.endswith(' A')].set_index(['model', 'test_set'])
    parity_diffs = [abs(float(a_rows.loc[index, metric]) - float(t08.loc[index, metric])) for index in a_rows.index for metric in METRIC_COLUMNS]
    assert max(parity_diffs) <= 0.001, f'Baseline A lệch T08: {max(parity_diffs)}'
    print('T08_BASELINE_PARITY_MAX_DIFF=', max(parity_diffs))
comparison_path = RESULT_DIR / 'comparison_B.csv'
comparison.to_csv(comparison_path, index=False, encoding='utf-8', lineterminator='\n')
comparison.to_csv(DRIVE_RESULT_DIR / 'comparison_B.csv', index=False, encoding='utf-8', lineterminator='\n')
display(comparison)
print('FINAL_TEST_COMPLETE=True; comparison_B=', comparison_path)

## 8. Kiểm tra AC, model card, CPU reload và zip Drive

Cell cuối không chọn lại model. Nó chỉ kiểm schema/kết quả đã có, tính mức giảm phone, ghi model card, nạp lại model trên CPU và tạo zip Drive.

In [ ]:
import transformers, zipfile
from datetime import date
assert transformers.__version__ == '5.18.0'
config_json = json.loads((LOCAL_B_DIR / 'config.json').read_text(encoding='utf-8'))
assert config_json['id2label'] == {'0': 'binh_thuong', '1': 'tieu_cuc'}
cpu_model = AutoModelForSequenceClassification.from_pretrained(LOCAL_B_DIR, local_files_only=True).to('cpu').eval()
cpu_tokenizer = AutoTokenizer.from_pretrained(LOCAL_B_DIR, local_files_only=True)
probe = normalize('Phim dài lê thê, xem mà mệt')
with torch.inference_mode():
    cpu_logits = cpu_model(**cpu_tokenizer(probe, return_tensors='pt', truncation=True, max_length=256)).logits
assert cpu_logits.shape == (1, 2)

lookup = comparison.set_index(['model', 'test_set'])
b_movie = lookup.loc[('PhoBERT B', 'movie')]
a_movie = lookup.loc[('PhoBERT A', 'movie')]
b_phone = lookup.loc[('PhoBERT B', 'phone')]
a_phone = lookup.loc[('PhoBERT A', 'phone')]
phone_drop = float(a_phone['f1_neg'] - b_phone['f1_neg'])
constant_movie = lookup.loc[('Constant positive', 'movie')]
ac02 = float(b_movie['f1_neg']) > float(a_movie['f1_neg'])

def metric_table(frame):
    lines = ['| Model | Config | Test | Precision neg | Recall neg | F1 neg | Macro-F1 |', '|---|:---:|:---:|---:|---:|---:|---:|']
    for row in frame.itertuples(index=False):
        lines.append(f'| {row.model} | {row.train_config} | {row.test_set} | {row.precision_neg:.6f} | {row.recall_neg:.6f} | {row.f1_neg:.6f} | {row.macro_f1:.6f} |')
    return '\n'.join(lines)

traditional_lines = []
for row in traditional_summary.itertuples(index=False):
    traditional_lines.append(f'- {row.model}: config `{row.train_config}`, validation macro-F1 `{row.macro_f1:.6f}`, F1 neg `{row.f1_neg:.6f}`.')
model_card = f'''# Model Card — PhoBERT B

## Phiên bản đề xuất

`phobert-B-{date.today().isoformat()}` — chỉ là phiên bản đề xuất cho T10; chưa thay model production trong T09.

## Nguồn khởi tạo và nhãn

- Fine-tune từ checkpoint research canonical `ml/artifacts/phobert_A_best/`, SHA-256 `{sha256(RESEARCH_CKPT / 'model.safetensors')}`; không dùng checkpoint backend.
- `id2label`: `0 = binh_thuong`, `1 = tieu_cuc`; `max_length=256`; threshold/argmax mặc định hai lớp.
- `normalize()` giống runtime, parity notebook 20/20. Không tách từ.

## Dữ liệu và chống leakage

- Movie train: 347 câu. Validation 15% phân tầng bằng seed 42: {len(movie_train_part)} train / {len(movie_validation)} validation.
- Config a chỉ dùng movie. Config b thêm mẫu phone train+dev có số dòng bằng movie (1:1), seed 42; phone test không thuộc pool lấy mẫu.
- `movie_test.csv` và phone `test.csv` chỉ được nạp sau marker `SELECTION_LOCKED`; không chọn lại sau khi xem test.

## Hyperparameter và lựa chọn bằng validation

- PhoBERT thử config a/b, learning rate 1e-5/2e-5, batch 8, AdamW, weight decay 0.01, warmup 10%, tối đa 5 epoch, early stopping patience 2.
- Chọn config `{phobert_selection['train_config']}`, learning rate `{phobert_selection['learning_rate']}`, epoch `{phobert_selection['best_epoch']}` vì đạt validation macro-F1 `{phobert_selection['validation_metrics']['macro_f1']:.6f}` và F1 neg `{phobert_selection['validation_metrics']['f1_neg']:.6f}`.
- Baseline truyền thống thử cùng grid TF-IDF/classifier của T01 trên cả a/b:
{chr(10).join(traditional_lines)}

## Kết quả final test

{metric_table(comparison)}

PhoBERT B {'thắng' if ac02 else 'không thắng'} PhoBERT A trên movie F1 tiêu cực: `{float(b_movie['f1_neg']):.6f}` so với `{float(a_movie['f1_neg']):.6f}`. Mức giảm trên phone so với A, tính `F1_A - F1_B`, là `{phone_drop:.6f}` (dương nghĩa là B thấp hơn A; âm nghĩa là B tốt hơn).

## Lệch nhãn và giới hạn

- Movie test có 149 câu, gồm 101 câu tiêu cực (67,8%). Baseline dự đoán hằng nhãn 1 đạt F1 neg `{float(constant_movie['f1_neg']):.6f}` nhưng macro-F1 chỉ `{float(constant_movie['macro_f1']):.6f}`; vì vậy phải đọc F1 neg cùng macro-F1.
- Movie test nhỏ và mất cân bằng; chênh lệch metric có độ bất định cao. Không dùng test này để tinh chỉnh tiếp.
- Dữ liệu phim chỉ 496 câu tổng cộng và bao gồm review phim/rạp. Model có thể chưa bao phủ tiếng lóng, châm biếm và câu khen/chê trộn lẫn.
- Fine-tune tiếp từ A có thể gây quên miền phone; mức giảm phone ở trên là trade-off thực nghiệm cần PO xem xét.
- Artifact chỉ được đề xuất cho T10 sau review/PO approval; T09 không thay checkpoint production.

## Tái lập và artifact

- Seed 42; manifest khóa selection: `selection_manifest.json`.
- Đã nạp lại trên CPU bằng `transformers==5.18.0`, logits có shape `(1, 2)`, `config.json` có đúng `id2label`.
- Drive: `MyDrive/thesis_results/phobert_B_best/` và `MyDrive/thesis_results/firefly_phobert_B.zip`. Model/tokenizer/zip không commit Git.
'''
model_card_path = PROJECT_ROOT / 'ml/model_card_B.md'
model_card_path.write_text(model_card, encoding='utf-8')
(DRIVE_RESULT_DIR / 'model_card_B.md').write_text(model_card, encoding='utf-8')
locked_manifest['test_loaded'] = True
locked_manifest['final_test_hashes'] = {'movie_test': sha256(MOVIE_TEST_PATH), 'phone_test': sha256(PHONE_TEST_PATH)}
locked_manifest['comparison_B_sha256'] = sha256(comparison_path)
write_json(LOCAL_B_DIR / 'selection_manifest.json', locked_manifest)
write_json(DRIVE_B_DIR / 'selection_manifest.json', locked_manifest)
drive_zip = Path(shutil.make_archive(str(DRIVE_ROOT / 'thesis_results/firefly_phobert_B'), 'zip', root_dir=DRIVE_B_DIR))
with zipfile.ZipFile(drive_zip) as archive:
    assert 'config.json' in archive.namelist() and 'model.safetensors' in archive.namelist()
evidence_paths = [DRIVE_B_DIR / 'config.json', DRIVE_B_DIR / 'model.safetensors', DRIVE_B_DIR / 'selection_manifest.json', drive_zip]
drive_evidence = {path.name: {'bytes': path.stat().st_size, 'sha256': sha256(path)} for path in evidence_paths}
print('CPU_RELOAD_OK transformers=', transformers.__version__)
print('AC_02_PHOBERT_B_GT_A_MOVIE=', ac02)
print('PHONE_F1_DROP_A_MINUS_B=', round(phone_drop, 6))
print('MODEL_CARD=', model_card_path)
print('DRIVE_MODEL=', DRIVE_B_DIR)
print('DRIVE_ZIP=', drive_zip)
print('DRIVE_ARTIFACT_EVIDENCE=', json.dumps(drive_evidence, indent=2))